# Prerequisites
본 `ipynb` 은 `Python=3.12` 에서 작성하였습니다. Package dependency 를 해결하기 위해 아래 cell 을 실행해주세요.

## Install Python packages

In [ ]:
%pip -q install -U agent-framework==1.0.0b260130

## Load environment variables from a .env file
secret 노출을 피하고 notebook 들간의 일관된 환경변수를 설정하기 위해 `dotenv` 을 이용한다.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

# Agent Skill

**Agent Skill** 이란 에이전트가 가질 수 있는 **재사용 가능한 역량 묶음**입니다. 하나의 스킬은 보통 다음으로 구성됩니다.

- **Instructions**: 해당 역량을 어떻게 사용할지에 대한 지시문
- **Tools**: 해당 역량을 수행하기 위한 도구 목록

여러 스킬을 조합하면 하나의 에이전트가 파일 조작, 메모리, 계산 등 다양한 일을 수행할 수 있습니다. 이 노트북에서는 스킬을 정의하고 조합하는 패턴을 다룹니다.

## Skill 구조 정의

스킬을 `name`, `instructions`, `tools` 를 가진 구조로 정의합니다. 딕셔너리나 dataclass 로 표현할 수 있습니다.

In [2]:
from dataclasses import dataclass
from typing import List, Any

@dataclass
class AgentSkill:
    """에이전트 스킬: instructions + tools 묶음"""
    name: str
    instructions: str
    tools: List[Any]

    def merge_instructions(self, other: "AgentSkill") -> str:
        """다른 스킬과 instructions 를 합칩니다."""
        return f"{self.instructions}\n\n{other.instructions}"

    def merge_tools(self, other: "AgentSkill") -> List[Any]:
        """다른 스킬과 tools 를 합칩니다. (중복 제거는 하지 않음)"""
        return [*self.tools, *other.tools]

def combine_skills(skills: List[AgentSkill]) -> tuple[str, List[Any]]:
    """여러 스킬을 하나의 instructions 와 tools 로 합칩니다."""
    if not skills:
        return "", []
    instructions = skills[0].instructions
    tools = list(skills[0].tools)
    for s in skills[1:]:
        instructions = instructions + "\n\n" + s.instructions
        tools.extend(s.tools)
    return instructions, tools

print("✅ AgentSkill 구조 및 combine_skills 유틸 정의 완료")

✅ AgentSkill 구조 및 combine_skills 유틸 정의 완료


## 스킬 1: 시간/계산 (Time & Calculator)

현재 시간 조회와 수식 계산 도구를 묶은 스킬입니다. (002-agentic-loop 에서 사용한 도구와 동일한 개념)

In [5]:
from datetime import datetime
from typing import Annotated
from pydantic import Field
from agent_framework import tool

@tool(name="get_current_time", description="현재 시간을 반환합니다.")
def get_current_time() -> str:
    return datetime.now().strftime("%Y년 %m월 %d일 %H시 %M분 %S초")

@tool(name="calculate", description="수학 계산을 수행합니다. 사칙연산과 괄호를 지원합니다.")
def calculate(
    expression: Annotated[str, Field(description="계산할 수식 (예: '2 + 3 * 4')")]
) -> str:
    try:
        allowed_chars = set("0123456789+-*/.() ")
        if not all(c in allowed_chars for c in expression):
            return "❌ 오류: 허용되지 않은 문자가 포함되어 있습니다."
        return str(eval(expression))
    except Exception as e:
        return f"❌ 계산 오류: {str(e)}"

skill_time_calc = AgentSkill(
    name="time_calculator",
    instructions="""
## 시간/계산 스킬
- 사용자가 현재 시간을 물어보면 get_current_time 을 사용하세요.
- 수식 계산 요청이 있으면 calculate 를 사용하세요. 수식만 전달하고 안전하게 계산하세요.
""".strip(),
    tools=[get_current_time, calculate],
)
print("✅ 스킬 정의:", skill_time_calc.name, "|", [t.name for t in skill_time_calc.tools])

✅ 스킬 정의: time_calculator | ['get_current_time', 'calculate']


## 스킬 2: 파일 읽기/나열 (File Read & List)

파일 내용 읽기와 디렉토리 나열 도구를 묶은 스킬입니다. (001-simple-code-assistant 의 read, ls 와 동일한 개념)

In [6]:
from pathlib import Path

@tool(name="read", description="파일 내용 읽기")
def read(file_path: Annotated[str, Field(description="파일 경로")]) -> str:
    try:
        path = Path(file_path).resolve()
        if not path.exists():
            return f"❌ 오류: 파일 '{file_path}'이(가) 존재하지 않습니다."
        if not path.is_file():
            return f"❌ 오류: '{file_path}'은(는) 파일이 아닙니다."
        content = path.read_text(encoding="utf-8")
        return f"✅ 파일 '{file_path}' 내용:\n{content}"
    except Exception as e:
        return f"❌ 파일 읽기 오류: {str(e)}"

@tool(name="ls", description="디렉토리 내용 나열")
def ls(
    dir_path: Annotated[str, Field(description="디렉토리 경로 (비어있으면 현재 디렉토리)")] = ""
) -> str:
    path = Path(dir_path).resolve() if dir_path else Path.cwd()
    if not path.exists():
        return f"❌ 오류: 디렉토리 '{dir_path}'이(가) 존재하지 않습니다."
    if not path.is_dir():
        return f"❌ 오류: '{dir_path}'은(는) 디렉토리가 아닙니다."
    try:
        items = []
        for item in sorted(path.iterdir()):
            item_type = "📁" if item.is_dir() else "📄"
            items.append(f"  {item_type} {item.name}" + ("/" if item.is_dir() else ""))
        return f"✅ 디렉토리 '{path}' 내용:\n" + "\n".join(items) if items else f"디렉토리 '{path}' 가 비어있습니다."
    except Exception as e:
        return f"❌ 디렉토리 나열 오류: {str(e)}"

skill_file = AgentSkill(
    name="file_read_list",
    instructions="""
## 파일 스킬
- 파일 내용을 보려면 read 도구를, 디렉토리 내용을 보려면 ls 도구를 사용하세요.
- 모든 경로는 워크스페이스(현재 디렉토리) 기준 상대 경로로 해석합니다.
""".strip(),
    tools=[read, ls],
)
print("✅ 스킬 정의:", skill_file.name, "|", [t.name for t in skill_file.tools])

✅ 스킬 정의: file_read_list | ['read', 'ls']


## 스킬 조합 및 ChatAgent 생성

위에서 정의한 두 스킬을 합쳐 하나의 에이전트를 만듭니다.

In [9]:
combined_instructions, combined_tools = combine_skills([skill_time_calc, skill_file])

base_instructions = """당신은 도움이 되는 AI 어시스턴트입니다.
사용자 요청에 맞는 도구를 선택해 사용하고, 결과를 명확히 전달하세요."""

full_instructions = base_instructions + "\n\n" + combined_instructions
print("==== 통합 instructions ====\n", full_instructions)
print("\n==== 통합 tools ====\n", [t.name for t in combined_tools])

==== 통합 instructions ====
 당신은 도움이 되는 AI 어시스턴트입니다.
사용자 요청에 맞는 도구를 선택해 사용하고, 결과를 명확히 전달하세요.

## 시간/계산 스킬
- 사용자가 현재 시간을 물어보면 get_current_time 을 사용하세요.
- 수식 계산 요청이 있으면 calculate 를 사용하세요. 수식만 전달하고 안전하게 계산하세요.

## 파일 스킬
- 파일 내용을 보려면 read 도구를, 디렉토리 내용을 보려면 ls 도구를 사용하세요.
- 모든 경로는 워크스페이스(현재 디렉토리) 기준 상대 경로로 해석합니다.

==== 통합 tools ====
 ['get_current_time', 'calculate', 'read', 'ls']


In [12]:
from agent_framework import ChatAgent
from agent_framework.openai import OpenAIResponsesClient

skill_agent = ChatAgent(
    chat_client=OpenAIResponsesClient(
        base_url=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
        model_id=AZURE_OPENAI_CHAT_DEPLOYMENT,
    ),
    name="SkillAgent",
    instructions=full_instructions,
    tools=combined_tools,
)

Agent 의 응답을 출력하는 함수를 정의합니다.

In [13]:
from agent_framework import Role

def print_result(result):
    """에이전트 실행 결과를 포맷팅하여 출력"""
    print("\n======= 🤖 Agent Messages =======")
    for idx, msg in enumerate(result.messages):
        if msg.role == Role.ASSISTANT:
            for content in msg.contents:
                if content.type == "text":
                    print(f"[{idx}] 💬 {content.text}")
                elif content.type == "function_call":
                    print(f"[{idx}] 🔧 {content.type} | {content.name} | {content.raw_representation.arguments}")
                else:
                    print(f"[{idx}] content type: {content.type}")
        elif msg.role == Role.TOOL:
            for content in msg.contents:
                print(f"[{idx}] 📤 {content.result}")
        else:
            print(f"[{idx}] role: {msg.role}")

### 테스트: 스킬 조합 에이전트 실행

시간/계산 스킬과 파일 스킬이 모두 사용 가능한지 확인합니다.

In [14]:
thread = skill_agent.get_new_thread()

result = await skill_agent.run(
    "지금 몇 시예요? 그리고 (10 + 20) * 3 은 얼마인지 계산해주세요.",
    thread=thread,
)
print_result(result)


======= 🤖 Agent Messages =======
[0] 🔧 function_call | get_current_time | {}
[0] 🔧 function_call | calculate | {"expression":"(10 + 20) * 3"}
[1] 📤 2026년 02월 21일 13시 24분 50초
[1] 📤 90
[2] 💬 지금 시간은 2026년 2월 21일 13시 24분 50초입니다.

수식 (10 + 20) * 3의 계산 결과는 90입니다.


In [ ]:
result = await skill_agent.run(
    "현재 디렉토리(.)에 뭐가 있는지 ls 로 확인해주고, 이 노트북 파일 004-agent-skill.ipynb 의 앞부분 몇 줄만 read 해주세요.",
    thread=thread,
)
print_result(result)

## 스킬만 따로 사용하는 에이전트

필요에 따라 스킬 하나만 부여한 에이전트를 만들 수 있습니다. 예: 시간/계산만 쓰는 에이전트.

In [15]:
time_only_agent = ChatAgent(
    chat_client=OpenAIResponsesClient(
        base_url=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
        model_id=AZURE_OPENAI_CHAT_DEPLOYMENT,
    ),
    name="TimeCalcAgent",
    instructions=base_instructions + "\n\n" + skill_time_calc.instructions,
    tools=skill_time_calc.tools,
)

t2 = time_only_agent.get_new_thread()
result = await time_only_agent.run("현재 시각과 100 * 99 를 알려주세요.", thread=t2)
print_result(result)


======= 🤖 Agent Messages =======
[0] 🔧 function_call | get_current_time | {}
[0] 🔧 function_call | calculate | {"expression":"100 * 99"}
[1] 📤 2026년 02월 21일 13시 25분 00초
[1] 📤 9900
[2] 💬 현재 시각은 2026년 02월 21일 13시 25분 00초입니다.
100 * 99의 계산 결과는 9900입니다.


## Wrap Up

### Agent Skill 요약

| 항목 | 설명 |
|------|------|
| **스킬 정의** | `instructions` + `tools` 를 하나의 단위(예: `AgentSkill`)로 묶기 |
| **스킬 조합** | 여러 스킬의 instructions 를 이어 붙이고, tools 리스트를 합쳐 하나의 ChatAgent 에 전달 |
| **확장** | 003 에서 다룬 메모리 도구를 `AgentSkill` 로 정의해 같은 방식으로 조합 가능 |

이렇게 스킬 단위로 나누어 두면, 도구와 지시문을 재사용하고 조합하기 쉬워집니다.